# Restaurant Branch Performance — Exploratory Data Analysis

A complete EDA on daily performance records across restaurant branches — covering summary statistics, descriptive and distribution analysis, correlation analysis, and comparisons across branches, regions, and store types. Key factors examined: customers, revenue, profit, marketing spend, staff count, delivery time, and customer ratings.

## 1. Loading and Understanding the Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Day13_Restaurant_Branch_Performance_Dataset.csv")
print("Dataset loaded successfully.")
df.head()

Dataset loaded successfully.


,Record_ID,Date,Day,Month,Branch,Region,Store_Type,Staff_Count,Marketing_Spend,Customers,...,Revenue,Food_Cost,Operating_Cost,Profit,Avg_Delivery_Min,Customer_Rating,Temperature_C,Humidity_Percent,Weather,Promotion
0,ST0001,2026-06-06,Saturday,June,Jaipur,North,Express,7,4977.25,172,...,66766.77,27954.29,46554.90,20211.87,29.0,4.44,20.8,70.3,Clear,NaN
1,ST0002,2026-02-10,Tuesday,February,Pune,West,Premium,14,8998.30,285,...,157230.89,63816.59,96813.95,60416.94,27.9,4.56,31.3,70.7,Clear,Loyalty Offer
2,ST0003,2026-03-07,Saturday,March,Delhi,North,Premium,16,10034.97,306,...,155811.96,54874.10,89888.91,65923.05,30.2,4.36,21.4,61.5,Clear,NaN
3,ST0004,2026-05-11,Monday,May,Mumbai,West,Standard,9,5095.84,198,...,80574.05,31429.44,53531.82,27042.23,21.7,4.65,22.9,40.4,Clear,NaN
4,ST0005,2026-01-14,Wednesday,January,Kochi,South,Express,9,6681.82,148,...,56579.89,21759.51,44376.25,12203.64,26.1,4.50,29.0,44.3,Clear,Combo Deal


In [2]:
print("Shape (rows, columns):", df.shape)
df.info()

Shape (rows, columns): (350, 22)
<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Record_ID         350 non-null    str    
 1   Date              350 non-null    str    
 2   Day               350 non-null    str    
 3   Month             350 non-null    str    
 4   Branch            350 non-null    str    
 5   Region            350 non-null    str    
 6   Store_Type        350 non-null    str    
 7   Staff_Count       350 non-null    int64  
 8   Marketing_Spend   350 non-null    float64
 9   Customers         350 non-null    int64  
 10  Orders            350 non-null    int64  
 11  Average_Bill      350 non-null    float64
 12  Revenue           350 non-null    float64
 13  Food_Cost         350 non-null    float64
 14  Operating_Cost    350 non-null    float64
 15  Profit            350 non-null    float64
 16  Avg_Delivery_Min  350 

In [3]:
df.isnull().sum()

Record_ID             0
Date                  0
Day                   0
Month                 0
Branch                0
Region                0
Store_Type            0
Staff_Count           0
Marketing_Spend       0
Customers             0
Orders                0
Average_Bill          0
Revenue               0
Food_Cost             0
Operating_Cost        0
Profit                0
Avg_Delivery_Min      0
Customer_Rating       0
Temperature_C         0
Humidity_Percent      0
Weather               0
Promotion           184
dtype: int64

`Promotion` has missing values for 184 out of 350 records — but looking at the data, this represents days with **no promotion running**, rather than a data quality issue. We'll treat it as its own category (`'No Promotion'`) instead of imputing it statistically.

In [4]:
df["Promotion"] = df["Promotion"].fillna("No Promotion")
df["Promotion"].value_counts()

Promotion
No Promotion     184
Weekend Offer     74
Combo Deal        59
Loyalty Offer     33
Name: count, dtype: int64

In [5]:
# Unique values in key categorical columns
for col in ["Branch", "Region", "Store_Type", "Weather"]:
    print(f"{col}: {sorted(df[col].unique().tolist())}")

Branch: ['Bengaluru', 'Delhi', 'Hyderabad', 'Jaipur', 'Kochi', 'Mumbai', 'Pune', 'Srinagar']
Region: ['North', 'South', 'West']
Store_Type: ['Express', 'Premium', 'Standard']
Weather: ['Clear', 'Cloudy', 'Hot', 'Rain']


## 2. Summary Statistics and Descriptive Analysis

In [6]:
df.describe()

,Staff_Count,Marketing_Spend,Customers,Orders,Average_Bill,Revenue,Food_Cost,Operating_Cost,Profit,Avg_Delivery_Min,Customer_Rating,Temperature_C,Humidity_Percent
count,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000
mean,11.957143,5516.347743,217.377143,187.054286,388.906371,86831.799657,31807.545114,56621.444486,30210.355171,26.098857,4.469314,27.740000,61.324571
std,4.338261,2050.906967,56.722524,50.558371,82.488196,36644.453901,13896.888942,18467.842215,19508.592192,4.388125,0.198258,4.903148,13.757322
min,5.000000,725.350000,77.000000,71.000000,180.000000,20540.760000,7262.010000,20782.380000,-6924.680000,13.900000,3.760000,14.500000,25.000000
25%,9.000000,4339.727500,180.250000,155.000000,337.315000,61261.717500,22326.662500,43797.670000,16915.492500,22.800000,4.320000,24.400000,51.325000
50%,11.000000,5393.370000,213.000000,182.500000,383.485000,79766.880000,28861.325000,53723.025000,26873.870000,26.100000,4.470000,27.800000,61.650000
75%,15.000000,6748.502500,255.750000,218.750000,441.765000,102976.185000,38370.370000,63961.227500,39823.742500,29.075000,4.600000,31.000000,70.375000
max,24.000000,11784.960000,360.000000,334.000000,623.280000,194018.920000,75946.590000,111489.140000,89096.190000,38.300000,4.960000,43.500000,95.000000


In [7]:
# Key business metrics at a glance
key_metrics = ["Customers", "Orders", "Revenue", "Profit", "Marketing_Spend",
               "Staff_Count", "Avg_Delivery_Min", "Customer_Rating"]

df[key_metrics].describe().round(2)

,Customers,Orders,Revenue,Profit,Marketing_Spend,Staff_Count,Avg_Delivery_Min,Customer_Rating
count,350.00,350.00,350.00,350.00,350.00,350.00,350.00,350.00
mean,217.38,187.05,86831.80,30210.36,5516.35,11.96,26.10,4.47
std,56.72,50.56,36644.45,19508.59,2050.91,4.34,4.39,0.20
min,77.00,71.00,20540.76,-6924.68,725.35,5.00,13.90,3.76
25%,180.25,155.00,61261.72,16915.49,4339.73,9.00,22.80,4.32
50%,213.00,182.50,79766.88,26873.87,5393.37,11.00,26.10,4.47
75%,255.75,218.75,102976.18,39823.74,6748.50,15.00,29.08,4.60
max,360.00,334.00,194018.92,89096.19,11784.96,24.00,38.30,4.96


In [8]:
for col in key_metrics:
    print(f"{col}: mean={df[col].mean():.2f}, median={df[col].median():.2f}, "
          f"min={df[col].min():.2f}, max={df[col].max():.2f}, std={df[col].std():.2f}")

Customers: mean=217.38, median=213.00, min=77.00, max=360.00, std=56.72
Orders: mean=187.05, median=182.50, min=71.00, max=334.00, std=50.56
Revenue: mean=86831.80, median=79766.88, min=20540.76, max=194018.92, std=36644.45
Profit: mean=30210.36, median=26873.87, min=-6924.68, max=89096.19, std=19508.59
Marketing_Spend: mean=5516.35, median=5393.37, min=725.35, max=11784.96, std=2050.91
Staff_Count: mean=11.96, median=11.00, min=5.00, max=24.00, std=4.34
Avg_Delivery_Min: mean=26.10, median=26.10, min=13.90, max=38.30, std=4.39
Customer_Rating: mean=4.47, median=4.47, min=3.76, max=4.96, std=0.20


## 3. Distribution Analysis

### 3.1 Revenue and Profit Distribution

In [9]:
print("Revenue distribution (Lakh currency units):")
print(df["Revenue"].describe())
print("\nProfit distribution:")
print(df["Profit"].describe())

# Profit margin as a derived metric
df["Profit_Margin_%"] = (df["Profit"] / df["Revenue"] * 100).round(2)
print("\nProfit Margin % distribution:")
print(df["Profit_Margin_%"].describe())

Revenue distribution (Lakh currency units):
count       350.000000
mean      86831.799657
std       36644.453901
min       20540.760000
25%       61261.717500
50%       79766.880000
75%      102976.185000
max      194018.920000
Name: Revenue, dtype: float64

Profit distribution:
count      350.000000
mean     30210.355171
std      19508.592192
min      -6924.680000
25%      16915.492500
50%      26873.870000
75%      39823.742500
max      89096.190000
Name: Profit, dtype: float64

Profit Margin % distribution:
count    350.000000
mean      31.403114
std       11.494707
min      -22.430000
25%       25.552500
50%       33.230000
75%       39.205000
max       50.570000
Name: Profit_Margin_%, dtype: float64


### 3.2 Customer Rating Distribution

In [10]:
print(df["Customer_Rating"].describe())
print("\nRating value counts (rounded to nearest 0.5):")
rounded_ratings = (df["Customer_Rating"] * 2).round() / 2
print(rounded_ratings.value_counts().sort_index())

count    350.000000
mean       4.469314
std        0.198258
min        3.760000
25%        4.320000
50%        4.470000
75%        4.600000
max        4.960000
Name: Customer_Rating, dtype: float64

Rating value counts (rounded to nearest 0.5):
Customer_Rating
4.0     47
4.5    270
5.0     33
Name: count, dtype: int64


### 3.3 Delivery Time Distribution

In [11]:
print(df["Avg_Delivery_Min"].describe())

# Categorize delivery speed
def delivery_speed(minutes):
    if minutes <= 20:
        return "Fast (<=20 min)"
    elif minutes <= 30:
        return "Moderate (21-30 min)"
    else:
        return "Slow (>30 min)"

df["Delivery_Speed_Category"] = df["Avg_Delivery_Min"].apply(delivery_speed)
df["Delivery_Speed_Category"].value_counts()

count    350.000000
mean      26.098857
std        4.388125
min       13.900000
25%       22.800000
50%       26.100000
75%       29.075000
max       38.300000
Name: Avg_Delivery_Min, dtype: float64


Delivery_Speed_Category
Moderate (21-30 min)    260
Slow (>30 min)           64
Fast (<=20 min)          26
Name: count, dtype: int64

### 3.4 Store Type and Region Distribution

In [12]:
print("Records by Store Type:")
print(df["Store_Type"].value_counts())
print("\nRecords by Region:")
print(df["Region"].value_counts())
print("\nRecords by Branch:")
print(df["Branch"].value_counts())

Records by Store Type:
Store_Type
Standard    196
Express     102
Premium      52
Name: count, dtype: int64

Records by Region:
Region
South    137
North    132
West      81
Name: count, dtype: int64

Records by Branch:
Branch
Srinagar     52
Jaipur       48
Kochi        48
Bengaluru    46
Mumbai       44
Hyderabad    43
Pune         37
Delhi        32
Name: count, dtype: int64


### 3.5 Weather and Promotion Distribution

In [13]:
print("Weather conditions:")
print(df["Weather"].value_counts())
print("\nPromotion types:")
print(df["Promotion"].value_counts())

Weather conditions:
Weather
Clear     189
Cloudy     82
Rain       48
Hot        31
Name: count, dtype: int64

Promotion types:
Promotion
No Promotion     184
Weekend Offer     74
Combo Deal        59
Loyalty Offer     33
Name: count, dtype: int64


## 4. Correlation Analysis

In [14]:
numeric_cols = ["Staff_Count", "Marketing_Spend", "Customers", "Orders", "Average_Bill",
                "Revenue", "Food_Cost", "Operating_Cost", "Profit", "Avg_Delivery_Min",
                "Customer_Rating", "Temperature_C", "Humidity_Percent"]

correlation_matrix = df[numeric_cols].corr().round(2)
correlation_matrix

,Staff_Count,Marketing_Spend,Customers,Orders,Average_Bill,Revenue,Food_Cost,Operating_Cost,Profit,Avg_Delivery_Min,Customer_Rating,Temperature_C,Humidity_Percent
Staff_Count,1.00,0.28,0.69,0.67,0.52,0.70,0.68,0.79,0.56,-0.18,0.08,0.08,-0.02
Marketing_Spend,0.28,1.00,0.41,0.42,0.33,0.43,0.40,0.49,0.35,0.10,-0.06,0.06,-0.05
Customers,0.69,0.41,1.00,0.98,0.53,0.88,0.84,0.86,0.84,0.18,-0.05,0.00,-0.10
Orders,0.67,0.42,0.98,1.00,0.52,0.86,0.83,0.84,0.82,0.19,-0.05,0.02,-0.09
Average_Bill,0.52,0.33,0.53,0.52,1.00,0.84,0.84,0.80,0.83,-0.03,0.04,-0.05,-0.04
Revenue,0.70,0.43,0.88,0.86,0.84,1.00,0.98,0.96,0.97,0.08,0.01,-0.03,-0.09
Food_Cost,0.68,0.40,0.84,0.83,0.84,0.98,1.00,0.97,0.92,0.08,0.00,-0.03,-0.08
Operating_Cost,0.79,0.49,0.86,0.84,0.80,0.96,0.97,1.00,0.86,0.02,0.02,0.01,-0.07
Profit,0.56,0.35,0.84,0.82,0.83,0.97,0.92,0.86,1.00,0.13,-0.01,-0.06,-0.11
Avg_Delivery_Min,-0.18,0.10,0.18,0.19,-0.03,0.08,0.08,0.02,0.13,1.00,-0.44,0.00,-0.11


In [15]:
# Correlation of each variable specifically with Revenue and Profit
print("Correlation with Revenue:")
print(correlation_matrix["Revenue"].sort_values(ascending=False))
print("\nCorrelation with Profit:")
print(correlation_matrix["Profit"].sort_values(ascending=False))

Correlation with Revenue:


Revenue             1.00
Food_Cost           0.98
Profit              0.97
Operating_Cost      0.96
Customers           0.88
Orders              0.86
Average_Bill        0.84
Staff_Count         0.70
Marketing_Spend     0.43
Avg_Delivery_Min    0.08
Customer_Rating     0.01
Temperature_C      -0.03
Humidity_Percent   -0.09
Name: Revenue, dtype: float64

Correlation with Profit:
Profit              1.00
Revenue             0.97
Food_Cost           0.92
Operating_Cost      0.86
Customers           0.84
Average_Bill        0.83
Orders              0.82
Staff_Count         0.56
Marketing_Spend     0.35
Avg_Delivery_Min    0.13
Customer_Rating    -0.01
Temperature_C      -0.06
Humidity_Percent   -0.11
Name: Profit, dtype: float64


In [16]:
# Correlation with Customer_Rating (what drives satisfaction?)
print("Correlation with Customer_Rating:")
print(correlation_matrix["Customer_Rating"].sort_values(ascending=False))

Correlation with Customer_Rating:
Customer_Rating     1.00
Staff_Count         0.08
Average_Bill        0.04
Operating_Cost      0.02
Humidity_Percent    0.02
Revenue             0.01
Food_Cost           0.00
Profit             -0.01
Customers          -0.05
Temperature_C      -0.05
Orders             -0.05
Marketing_Spend    -0.06
Avg_Delivery_Min   -0.44
Name: Customer_Rating, dtype: float64


In [17]:
# Strongest relationships overall (excluding self-correlation of 1.0)
corr_pairs = correlation_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs != 1.0]
corr_pairs = corr_pairs.abs().sort_values(ascending=False)
print("Top 10 strongest correlations (absolute value):")
corr_pairs.head(20)[::2]

Top 10 strongest correlations (absolute value):


Customers       Orders            0.98
Revenue         Food_Cost         0.98
Food_Cost       Operating_Cost    0.97
Operating_Cost  Food_Cost         0.97
                Revenue           0.96
Food_Cost       Profit            0.92
Customers       Revenue           0.88
Operating_Cost  Customers         0.86
Orders          Revenue           0.86
Profit          Operating_Cost    0.86
dtype: float64

## 5. Comparing Performance Across Branches

In [18]:
branch_summary = df.groupby("Branch").agg(
    Total_Revenue=("Revenue", "sum"),
    Avg_Revenue=("Revenue", "mean"),
    Total_Profit=("Profit", "sum"),
    Avg_Profit_Margin=("Profit_Margin_%", "mean"),
    Avg_Customers=("Customers", "mean"),
    Avg_Rating=("Customer_Rating", "mean"),
    Avg_Delivery_Min=("Avg_Delivery_Min", "mean")
).round(2).sort_values(by="Total_Revenue", ascending=False)

branch_summary

,Total_Revenue,Avg_Revenue,Total_Profit,Avg_Profit_Margin,Avg_Customers,Avg_Rating,Avg_Delivery_Min
Branch,,,,,,,
Srinagar,4510816.59,86746.47,1575751.62,31.69,214.73,4.48,25.97
Bengaluru,4173677.46,90732.12,1466982.08,31.11,227.50,4.47,26.56
Jaipur,4163815.60,86746.16,1462545.76,32.50,223.27,4.47,26.58
Kochi,4107581.66,85574.62,1387297.74,30.24,209.06,4.42,25.84
Hyderabad,3818610.68,88804.90,1350328.01,31.99,220.79,4.47,25.52
Mumbai,3554286.95,80779.25,1169563.49,29.91,213.77,4.44,26.86
Pune,3259977.12,88107.49,1156826.16,32.20,214.92,4.54,25.53
Delhi,2802363.82,87573.87,1004329.45,31.80,213.97,4.49,25.69


## 6. Comparing Performance Across Regions

In [19]:
region_summary = df.groupby("Region").agg(
    Total_Revenue=("Revenue", "sum"),
    Avg_Revenue=("Revenue", "mean"),
    Total_Profit=("Profit", "sum"),
    Avg_Profit_Margin=("Profit_Margin_%", "mean"),
    Avg_Marketing_Spend=("Marketing_Spend", "mean"),
    Avg_Customer_Rating=("Customer_Rating", "mean")
).round(2).sort_values(by="Total_Revenue", ascending=False)

region_summary

,Total_Revenue,Avg_Revenue,Total_Profit,Avg_Profit_Margin,Avg_Marketing_Spend,Avg_Customer_Rating
Region,,,,,,
South,12099869.80,88320.22,4204607.83,31.08,5513.23,4.45
North,11476996.01,86946.94,4042626.83,32.01,5477.53,4.48
West,6814264.07,84126.72,2326389.65,30.96,5584.88,4.49


## 7. Comparing Performance Across Store Types

In [20]:
store_type_summary = df.groupby("Store_Type").agg(
    Number_of_Records=("Record_ID", "count"),
    Avg_Revenue=("Revenue", "mean"),
    Avg_Profit=("Profit", "mean"),
    Avg_Profit_Margin=("Profit_Margin_%", "mean"),
    Avg_Staff_Count=("Staff_Count", "mean"),
    Avg_Delivery_Min=("Avg_Delivery_Min", "mean"),
    Avg_Customer_Rating=("Customer_Rating", "mean")
).round(2).sort_values(by="Avg_Revenue", ascending=False)

store_type_summary

,Number_of_Records,Avg_Revenue,Avg_Profit,Avg_Profit_Margin,Avg_Staff_Count,Avg_Delivery_Min,Avg_Customer_Rating
Store_Type,,,,,,,
Premium,52,152867.31,62890.24,40.80,17.60,26.40,4.46
Standard,196,87268.67,30630.79,34.15,12.54,25.98,4.47
Express,102,52327.16,12742.14,21.34,7.96,26.17,4.46


## 8. Impact of Marketing Spend, Promotions, and Weather

In [21]:
# Does higher marketing spend consistently bring more customers?
marketing_bins = pd.qcut(df["Marketing_Spend"], q=4, labels=["Low", "Medium", "High", "Very High"])
df["Marketing_Spend_Tier"] = marketing_bins

df.groupby("Marketing_Spend_Tier", observed=True).agg(
    Avg_Customers=("Customers", "mean"),
    Avg_Revenue=("Revenue", "mean"),
    Avg_Profit=("Profit", "mean")
).round(2)

,Avg_Customers,Avg_Revenue,Avg_Profit
Marketing_Spend_Tier,,,
Low,194.23,73811.37,25024.82
Medium,209.00,79346.21,27434.90
High,214.90,82843.16,27046.25
Very High,251.26,111196.07,41267.96


In [22]:
# Effect of promotions on customers and revenue
df.groupby("Promotion").agg(
    Avg_Customers=("Customers", "mean"),
    Avg_Revenue=("Revenue", "mean"),
    Avg_Profit=("Profit", "mean")
).round(2).sort_values(by="Avg_Revenue", ascending=False)

,Avg_Customers,Avg_Revenue,Avg_Profit
Promotion,,,
Loyalty Offer,226.97,91937.89,32346.09
Weekend Offer,224.54,88808.72,31730.63
Combo Deal,218.64,86269.73,29779.94
No Promotion,212.37,85301.20,29353.92


In [23]:
# Effect of weather on customers and delivery time
df.groupby("Weather").agg(
    Avg_Customers=("Customers", "mean"),
    Avg_Revenue=("Revenue", "mean"),
    Avg_Delivery_Min=("Avg_Delivery_Min", "mean")
).round(2).sort_values(by="Avg_Customers", ascending=False)

,Avg_Customers,Avg_Revenue,Avg_Delivery_Min
Weather,,,
Rain,226.85,92974.88,25.95
Clear,216.24,84720.81,25.89
Cloudy,216.10,87724.00,26.24
Hot,213.03,87830.13,27.23


## 9. Staff Count vs. Operational Efficiency

In [24]:
# Does more staff mean faster delivery or better ratings?
staff_bins = pd.cut(df["Staff_Count"], bins=[0, 8, 14, 25], labels=["Small (<=8)", "Medium (9-14)", "Large (>14)"])
df["Staff_Size_Category"] = staff_bins

df.groupby("Staff_Size_Category", observed=True).agg(
    Avg_Delivery_Min=("Avg_Delivery_Min", "mean"),
    Avg_Customer_Rating=("Customer_Rating", "mean"),
    Avg_Revenue=("Revenue", "mean"),
    Avg_Profit_Margin=("Profit_Margin_%", "mean")
).round(2)

,Avg_Delivery_Min,Avg_Customer_Rating,Avg_Revenue,Avg_Profit_Margin
Staff_Size_Category,,,,
Small (<=8),26.83,4.47,58251.36,26.91
Medium (9-14),26.49,4.44,84801.89,32.30
Large (>14),24.81,4.51,115310.62,33.86


## 10. Best and Worst Performing Days

In [25]:
# Top 5 highest revenue days
df.sort_values(by="Revenue", ascending=False).head()[
    ["Record_ID", "Branch", "Date", "Revenue", "Profit", "Customers", "Promotion"]]

,Record_ID,Branch,Date,Revenue,Profit,Customers,Promotion
85,ST0086,Srinagar,2026-04-28,194018.92,82529.78,301,Weekend Offer
246,ST0247,Srinagar,2026-05-15,193782.79,87017.17,339,Weekend Offer
323,ST0324,Delhi,2026-03-28,191425.08,83745.96,320,No Promotion
204,ST0205,Bengaluru,2026-04-19,187875.32,80422.67,360,Loyalty Offer
333,ST0334,Bengaluru,2026-01-04,186854.25,79823.25,360,Weekend Offer


In [26]:
# Bottom 5 lowest profit margin days
df.sort_values(by="Profit_Margin_%", ascending=True).head()[
    ["Record_ID", "Branch", "Date", "Revenue", "Profit_Margin_%", "Weather"]]

,Record_ID,Branch,Date,Revenue,Profit_Margin_%,Weather
13,ST0014,Jaipur,2026-04-22,30872.58,-22.43,Clear
56,ST0057,Srinagar,2026-05-07,33108.79,-13.17,Cloudy
39,ST0040,Mumbai,2026-04-30,23560.23,-12.49,Clear
299,ST0300,Pune,2026-02-09,24442.30,-6.57,Cloudy
81,ST0082,Bengaluru,2026-01-07,29495.71,-5.35,Clear


## 11. Key Observations

Based on the complete exploratory analysis above:

1. **Revenue is driven mainly by volume metrics, not marketing spend alone.** `Revenue` correlates very strongly with `Customers` (~0.88), `Orders` (~0.86), and `Average_Bill` (~0.84), while its correlation with `Marketing_Spend` is only moderate (~0.43) — marketing spend helps, but customer footfall and order value are the bigger levers behind revenue.

2. **Profit moves almost in lockstep with revenue.** `Profit` correlates extremely strongly with `Revenue` (~0.97), confirming that on this dataset, cost structure scales fairly predictably with revenue rather than eroding margins unpredictably at higher volumes.

3. **Delivery speed is one of the more meaningful drivers of customer satisfaction.** `Customer_Rating` shows almost no correlation with revenue, profit, or marketing spend (all near 0), but a clear moderate negative correlation with `Avg_Delivery_Min` (~-0.44) — slower deliveries are the strongest numeric predictor of lower ratings in this dataset.

4. **Store type has a major effect on revenue, but almost no effect on delivery speed.** Premium stores generate roughly 3x the average revenue of Express stores, yet average delivery time is nearly identical (~26 minutes) across all three store types — so delivery speed here is not simply a function of store format.

5. **Staff count doesn't translate strongly into better ratings.** The correlation between `Staff_Count` and `Customer_Rating` is very weak (~0.08), suggesting that simply adding more staff has limited direct impact on customer experience — other factors like delivery time matter more.

6. **Promotions modestly lift customer volume and revenue, but the effect is smaller than expected.** Days with a Loyalty Offer or Weekend Offer show somewhat higher average customers and revenue than days with no promotion, but the gap is fairly small — suggesting promotions provide a mild, not dramatic, uplift.

7. **Regional performance is fairly close, with the South slightly ahead.** Average revenue is highest in the South, followed closely by the North, with the West trailing — the gap between regions is real but not dramatic, unlike the much larger spread seen across store types.

8. **Marketing spend shows a real but moderate relationship with customer traffic.** Grouping records into marketing-spend tiers shows customers and revenue trending upward from the lowest to highest spend tier, supporting marketing as a contributing factor — just not as dominant a factor as customer volume itself.